In [1]:
import os 
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
open_ai_api_key = os.getenv("OPENAI_API_KEY")

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

# Load the document from the specified file using the TextLoader
loader = TextLoader("data\\state_of_the_union.txt", encoding="utf-8")
loaded_document = loader.load()

# Split the loaded document into smaller chunks for easier processing
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks_of_text = text_splitter.split_documents(loaded_document)

# Create a FAISS vector store from the document chunks  using OpenAI embeddings
embeddings = OpenAIEmbeddings()
vector_db = FAISS.from_documents(chunks_of_text, embeddings)

# Create a retriever from the vector database
retriever = vector_db.as_retriever()

In [6]:
question = "What did he say about kentaji brown jackson ?"
response = retriever.invoke(question)
response

[Document(id='37ac07aa-c9c3-4ca0-8bca-66c2c76c8624', metadata={'source': 'data\\state_of_the_union.txt'}, page_content='Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'),
 Document(id='5b0a26a3-46c1-42f3-b77c-66d3027b44b3', metadata={'source': 'data\\sta

In [8]:
len(response)

4

In [9]:
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

In [10]:
response = retriever.invoke(question)
response

[Document(id='37ac07aa-c9c3-4ca0-8bca-66c2c76c8624', metadata={'source': 'data\\state_of_the_union.txt'}, page_content='Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'),
 Document(id='5b0a26a3-46c1-42f3-b77c-66d3027b44b3', metadata={'source': 'data\\sta

## Simple use with LCEL

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
template = """Answer the question based only on the following context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI() 

In [ ]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# Prepare the retriever context for the chain
retriever_context = {"context": retriever | format_docs, "question": RunnablePassthrough()}

# Create the chain that will process the context and generate the output
chain = (
    retriever_context
    | prompt 
    | model
    | StrOutputParser()
)

In [21]:
response = chain.invoke(question)

In [22]:
response

"He mentioned that Ketanji Brown Jackson was nominated 4 days ago and is one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence."

In [19]:
type(response)

str